In [2]:
import os
import json
import re
from pathlib import Path
from typing import Dict, List, Set, Optional

from tqdm import tqdm

from qiskit import transpile
from qiskit_aer import AerSimulator
from qiskit_ibm_runtime import QiskitRuntimeService, Session, SamplerV2 as Sampler
from qiskit_ibm_runtime.runtime_job import (
    RuntimeJobMaxTimeoutError,
    RuntimeJobFailureError,
    RuntimeInvalidStateError,
)


def simulate_and_store_z_expectations_real_qpu_json(
    qpy_folder: str,
    output_json_path: str,
    shots: int = 1024,
    per_circuit_timeout_s: float = 60.0,
    backend_name: str = "ibm_fez",
    service: Optional[QiskitRuntimeService] = None,
) -> None:
    """
    Like before but:
      * Collects first 10 circuits.
      * Runs their ideal AerSimulator ⟨Z⟩ immediately.
      * Submits all 10 hardware jobs in parallel (one per circuit), records job IDs.
      * Waits for completion and retrieves hardware results via job handles / job IDs.
      * Writes per-circuit shot-level JSONs (ideal + hardware) and a summary JSON with
        z_ideal, z_hardware, and hardware_job_id.
    """

    def counts_to_z_expectation(counts: Dict[str, int], n_qubits: int) -> List[float]:
        total = sum(counts.values())
        if total == 0:
            return [0.0] * n_qubits
        exp = [0.0] * n_qubits
        for key, cnt in counts.items():
            if key.startswith(("0x", "0X")):
                val = int(key, 16)
            else:
                val = int(key, 2)
            bits = bin(val)[2:].zfill(n_qubits)[::-1]  # little-endian
            for i in range(n_qubits):
                z = 1 if bits[i] == "0" else -1
                exp[i] += z * cnt
        return [round(e / total, 6) for e in exp]

    def unique_key(preferred: str, fallback_base: str, taken: Set[str]) -> str:
        base = preferred.strip() if preferred and preferred.strip() else fallback_base
        if base not in taken:
            taken.add(base)
            return base
        k = 2
        while True:
            cand = f"{base}#{k}"
            if cand not in taken:
                taken.add(cand)
                return cand
            k += 1

    def has_measurements(qc) -> bool:
        return any(inst.name == "measure" for inst, _, _ in qc.data)

    def sanitize(name: str) -> str:
        return re.sub(r"[^A-Za-z0-9._-]", "_", name)

    if service is None:
        service = QiskitRuntimeService()  # assumes auth already configured

    # get the backend object
    backends = service.backends(name=backend_name)
    if not backends:
        raise RuntimeError(f"No backend found with name '{backend_name}'")
    backend_obj = backends[0]

    ideal_sim = AerSimulator()

    # Prepare output structures
    results_dict: Dict[str, Dict[str, object]] = {}
    taken_names: Set[str] = set()

    output_base = Path(output_json_path)
    shot_dir = output_base.parent / (output_base.stem + "_shot_data")
    shot_dir.mkdir(parents=True, exist_ok=True)

    # We'll collect up to 10 circuits with their metadata and ideal results first
    hw_queue: List[Dict] = []  # each entry: {circuit_key, safe_key, n_qubits, tqc_hw, z_ideal}

    # === Phase 1: gather first 10 circuits, run ideal, prepare hardware-transpiled circuits ===
    for file in tqdm(sorted(os.listdir(qpy_folder)), desc="Scanning QPYs"):
        if len(hw_queue) >= 10:
            break
        if not file.endswith(".qpy"):
            continue
        file_path = os.path.join(qpy_folder, file)
        try:
            with open(file_path, "rb") as f:
                from qiskit import qpy

                circuits = qpy.load(f)
        except Exception as e:
            print(f"⚠️ Error loading {file}: {e}")
            continue

        for idx, qc in enumerate(circuits):
            if len(hw_queue) >= 10:
                break
            n_qubits = qc.num_qubits
            # circuit_key = unique_key(qc.name, f"{Path(file).stem}_{idx}", taken_names)
            # safe_key = sanitize(circuit_key)
            circuit_key = file
            safe_key = file

            # === Ideal (Aer) run ===
            qc_ideal = qc.copy()
            if not has_measurements(qc_ideal):
                qc_ideal = qc_ideal.copy()
                qc_ideal.measure_all()
            try:
                tqc_ideal = transpile(qc_ideal, backend=ideal_sim)
                job_ideal = ideal_sim.run(tqc_ideal, shots=1, memory=True)
                ideal_result = job_ideal.result()
                ideal_counts = ideal_result.get_counts()
                z_ideal = counts_to_z_expectation(ideal_counts, n_qubits)
                try:
                    ideal_bitstrings = ideal_result.get_memory()
                except Exception:
                    ideal_bitstrings = []
                    for b, c in ideal_counts.items():
                        ideal_bitstrings.extend([b] * c)
                ideal_payload = {
                    "z": z_ideal,
                    "counts": ideal_counts,
                    "bitstrings": ideal_bitstrings,
                }
                with open(shot_dir / f"{safe_key}__ideal_shots.json", "w") as f_out:
                    json.dump(ideal_payload, f_out, indent=2)
            except Exception as e:
                print(f"❌ Aer (ideal) failed for '{circuit_key}': {e}")
                continue  # skip enqueueing this circuit

            # === Prepare hardware (transpiled) circuit ===
            qc_hw = qc.copy()
            if not has_measurements(qc_hw):
                qc_hw = qc_hw.copy()
                qc_hw.measure_all()
            try:
                tqc_hw = transpile(qc_hw, backend=backend_obj)
            except Exception as e:
                print(f"❌ Transpile-to-hardware failed for '{circuit_key}': {e}")
                continue

            hw_queue.append(
                {
                    "circuit_key": circuit_key,
                    "safe_key": safe_key,
                    "n_qubits": n_qubits,
                    "tqc_hw": tqc_hw,
                    "z_ideal": z_ideal,
                }
            )

    if not hw_queue:
        print("No circuits prepared for hardware execution.")
        return

    # === Phase 2: submit all hardware jobs at once ===
    with Session(backend_obj) as session:  # session groups/prioritizes. :contentReference[oaicite:0]{index=0}
        sampler = Sampler(mode=session)  # session-mode SamplerV2. 

        # Submit jobs
        jobs_info = []  # list of dicts with job handle and metadata
        for entry in hw_queue:
            circuit_key = entry["circuit_key"]
            safe_key = entry["safe_key"]
            n_qubits = entry["n_qubits"]
            tqc_hw = entry["tqc_hw"]
            z_ideal = entry["z_ideal"]

            try:
                job_hw = sampler.run([tqc_hw], shots=shots)
                hw_job_id = job_hw.job_id()  # record job id for sanity check. :contentReference[oaicite:2]{index=2}
                jobs_info.append(
                    {
                        "circuit_key": circuit_key,
                        "safe_key": safe_key,
                        "n_qubits": n_qubits,
                        "job_hw": job_hw,
                        "z_ideal": z_ideal,
                        "hardware_job_id": hw_job_id,
                    }
                )
                print(f"Submitted hardware job for '{circuit_key}' id={hw_job_id}")
            except Exception as e:
                print(f"❌ Submission failed for '{circuit_key}': {e}")
                continue

        # === Phase 3: wait & collect results per job ===
        for job_entry in jobs_info:
            circuit_key = job_entry["circuit_key"]
            safe_key = job_entry["safe_key"]
            n_qubits = job_entry["n_qubits"]
            job_hw = job_entry["job_hw"]
            z_ideal = job_entry["z_ideal"]
            hardware_job_id = job_entry["hardware_job_id"]

            try:
                pub_result = job_hw.result(timeout=per_circuit_timeout_s)[0]  # blocking wait. :contentReference[oaicite:3]{index=3}
                # Preferred way to get combined counts for SamplerV2 result. :contentReference[oaicite:4]{index=4}
                try:
                    hw_counts = pub_result.data.meas.get_counts()
                except Exception:
                    try:
                        hw_counts = pub_result.join_data().get_counts()
                    except Exception:
                        hw_counts = {}
                if not hw_counts:
                    print(f"⚠️ No counts extracted for hardware run '{circuit_key}'; skipping.")
                    continue
                z_hardware = counts_to_z_expectation(hw_counts, n_qubits)

                # attempt to get per-shot bitstrings (preserves some order if available)
                try:
                    hw_bitstrings = pub_result.data.meas.get_strings()
                except Exception:
                    hw_bitstrings = []
                    if isinstance(hw_counts, dict):
                        for b, c in hw_counts.items():
                            hw_bitstrings.extend([b] * c)

                hw_payload = {
                    "z": z_hardware,
                    "counts": hw_counts,
                    "bitstrings": hw_bitstrings,
                }
                with open(shot_dir / f"{safe_key}__hardware_shots.json", "w") as f_out:
                    json.dump(hw_payload, f_out, indent=2)

                # final summary entry
                results_dict[circuit_key] = {
                    "z_ideal": z_ideal,
                    "z_hardware": z_hardware,
                    "hardware_job_id": hardware_job_id,
                }

            except RuntimeJobMaxTimeoutError as e:
                print(f"⏱️ Hardware timeout for '{circuit_key}' (job {hardware_job_id}): {e}")
            except (RuntimeJobFailureError, RuntimeInvalidStateError) as e:
                print(f"❌ Hardware job error for '{circuit_key}' (job {hardware_job_id}): {e}")
            except Exception as e:
                print(f"❌ Unexpected error retrieving hardware result for '{circuit_key}' (job {hardware_job_id}): {e}")

    # === Phase 4: persist summary JSON ===
    try:
        with open(output_json_path, "w") as out_f:
            json.dump(results_dict, out_f, indent=2)
        print(f"\n✅ Aggregated summary saved to {output_json_path}")
        print(f"📁 Shot-level details under {shot_dir}")
    except Exception as e:
        print(f"❌ Failed to write summary JSON: {e}")


In [4]:
simulate_and_store_z_expectations_real_qpu_json(
    qpy_folder="../../../../../quantum/ExecutionResults/StoredCircuits/",
    output_json_path="real_z_expectations.json",
    shots=1024,
    per_circuit_timeout_s=1000
    # service=service
)

Scanning QPYs:   0%|                                                                                          | 0/7004 [00:00<?, ?it/s]/tmp/ipykernel_117810/2287333576.py:67: DeprecationWarning: Treating CircuitInstruction as an iterable is deprecated legacy behavior since Qiskit 1.2, and will be removed in Qiskit 3.0. Instead, use the `operation`, `qubits` and `clbits` named attributes.
  return any(inst.name == "measure" for inst, _, _ in qc.data)
Scanning QPYs:   0%|                                                                                 | 10/7004 [00:04<54:50,  2.13it/s]


Submitted hardware job for '00033f13-0596-463f-b3ff-efdf4912e7da.qpy' id=d25pjguliavc738jh1j0
Submitted hardware job for '00195490-b4cb-41ce-b058-34eeb8534036.qpy' id=d25pjh6liavc738jh1jg
Submitted hardware job for '001d6dcc-6693-451b-9201-0b173e00b266.qpy' id=d25pjheliavc738jh1l0
Submitted hardware job for '0029dbf3-1e47-49cf-abb7-a89ae132cce9.qpy' id=d25pjheliavc738jh1lg
Submitted hardware job for '00333e29-9e8e-4d2c-ad06-82f8582679b7.qpy' id=d25pjhn291fs73962th0
Submitted hardware job for '003541aa-50cd-4f90-b262-70d6f393c56e.qpy' id=d25pjhuliavc738jh1m0
Submitted hardware job for '0038ebf2-bc92-4606-9145-f64584c17f63.qpy' id=d25pji739ohc73e791e0
Submitted hardware job for '003e2d72-1bf2-48c4-883f-a52398b6fc28.qpy' id=d25pjidc17vc73du2q40
Submitted hardware job for '00419ae5-9ec3-4f35-acaa-388c25636ee9.qpy' id=d25pjif291fs73962ti0
Submitted hardware job for '00465a4e-5fbd-4384-9972-817a9618257a.qpy' id=d25pjin39ohc73e791f0
❌ Unexpected error retrieving hardware result for '00033f13-

KeyboardInterrupt: 